In [ ]:
install.packages("lavaan")
library(lavaan)
library(readxl)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘mnormt’, ‘pbivnorm’, ‘numDeriv’, ‘quadprog’


This is lavaan 0.7-3
lavaan is FREE software! Please report any bugs.



In [ ]:
path <- list.files("/content/", pattern = "Responses.*\\.xlsx$", recursive = TRUE, full.names = TRUE)[1]
raw <- read_excel(path)
raw <- raw[as.Date(raw$Timestamp) <= as.Date("2026-07-15"), ]

age <- gsub("–", "-", raw[[3]])
items <- raw[, 7:26]
df <- as.data.frame(lapply(items, function(x) as.integer(sub(".*\\((\\d)\\).*", "\\1", x))))
names(df) <- sub(".*\\[(\\w+)\\..*", "\\1", names(items))
df <- df[age != "Below 18", ]

cat("Analysed sample:", nrow(df), "\n")
scale_mean <- function(p) rowMeans(df[, grep(paste0("^", p, "[0-9]$"), names(df))])
round(c(employability = mean(scale_mean("EMP")), skill_relevance = mean(scale_mean("SR")),
        income_expect = mean(scale_mean("IE")), career_sat = mean(scale_mean("CS"))), 3)

Analysed sample: 380 


employability skill_relevance   income_expect      career_sat 
          3.972           3.967           4.276           4.484

In [ ]:
model <- '
employability   =~ EMP1 + EMP2 + EMP3 + EMP4 + EMP5
skill_relevance =~ SR1 + SR2 + SR3 + SR4 + SR5
income_expect   =~ IE1 + IE2 + IE3 + IE4 + IE5
career_sat      =~ CS1 + CS2 + CS3 + CS4 + CS5
career_sat ~ employability + skill_relevance + income_expect
'
paths <- function(fit) {
  s <- standardizedSolution(fit)
  s[s$op == "~", c("lhs", "rhs", "est.std", "se", "z", "pvalue")]
}

In [ ]:
fit_wlsmv <- sem(model, data = df, ordered = names(df), estimator = "WLSMV")
round(fitMeasures(fit_wlsmv, c("chisq.scaled", "df.scaled", "pvalue.scaled",
                               "cfi.scaled", "tli.scaled", "rmsea.scaled", "srmr")), 3)
paths(fit_wlsmv)
lavInspect(fit_wlsmv, "rsquare")["career_sat"]

chisq.scaled     df.scaled pvalue.scaled    cfi.scaled    tli.scaled 
      146.768       164.000         0.829         1.000         1.001 
 rmsea.scaled          srmr 
        0.000         0.027

,lhs,rhs,est.std,se,z,pvalue
,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>
21,career_sat,employability,0.3747625,0.08885989,4.217454,2.470766e-05
22,career_sat,skill_relevance,0.3295601,0.07074555,4.658386,3.186986e-06
23,career_sat,income_expect,0.3287749,0.07011858,4.688841,2.747565e-06


career_sat 
 0.8866405

In [ ]:
fit_mlr <- sem(model, data = df, estimator = "MLR")
round(fitMeasures(fit_mlr, c("chisq.scaled", "df.scaled", "pvalue.scaled",
                             "cfi.robust", "tli.robust", "rmsea.robust", "srmr")), 3)
paths(fit_mlr)
lavInspect(fit_mlr, "rsquare")["career_sat"]

chisq.scaled     df.scaled pvalue.scaled    cfi.robust    tli.robust 
      159.986       164.000         0.574         1.000         1.001 
 rmsea.robust          srmr 
        0.000         0.021

,lhs,rhs,est.std,se,z,pvalue
,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>
21,career_sat,employability,0.3091617,0.05620384,5.500722,3.782385e-08
22,career_sat,skill_relevance,0.1880485,0.04063959,4.627224,3.705994e-06
23,career_sat,income_expect,0.4466244,0.05236476,8.529104,0.000000e+00


career_sat 
 0.6944103

In [ ]:
fit_mlm <- sem(model, data = df, estimator = "MLM")
round(fitMeasures(fit_mlm, c("chisq", "chisq.scaled", "df", "pvalue.scaled",
                             "chisq.scaling.factor")), 3)

chisq         chisq.scaled                   df 
             169.808              157.076              164.000 
       pvalue.scaled chisq.scaling.factor 
               0.637                1.081